1. Import


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from pathlib import Path

In [13]:
df = pd.read_csv(r"G:\Recommandation-de-films\notebooks\02-preprocessing\df_films_clean.csv")

Je vais partir sur un filtre de seuil minimum de votes (Bayesian average / IMDB Weighted Rating),

afin de ne pas surnoter des films notés 5 par une seule personne et garder une cohérence de vote.

La formule est : (WR) = (v ÷ (v+m)) × R + (m ÷ (v+m)) × C



In [7]:
C = df['averageRating'].mean()
m = df['numVotes'].quantile(0.95)

# Avec un quantile à 0.90, le nombre de votes minimal est de 1 835
# À 0.95, le nombre de votes minimal est de 5 845
# À 0.99, le nombre de votes minimal est de 73 566
# Nous pouvons faire varier le nombre minimal requis pour figurer dans la liste,
# ce qui peut être intéressant selon le type de recommandations que nous souhaitons faire

df['weight_rating'] = (
    (df['numVotes'] / (df['numVotes'] + m)) * df['averageRating'] +
    (m / (df['numVotes'] + m)) * C)


top_films = df.sort_values('weight_rating', ascending=False)[['primaryTitle', 'startYear', 'genres', 'averageRating', 'numVotes', 'weight_rating']].head(20)
top_films

,primaryTitle,startYear,genres,averageRating,numVotes,weight_rating
62313,The Shawshank Redemption,1994.0,Drama,9.3,3175959.0,9.294222
37329,The Godfather,1972.0,"Crime,Drama",9.2,2218556.0,9.191997
139895,The Dark Knight,2008.0,"Action,Crime,Drama",9.1,3155324.0,9.094554
79867,The Lord of the Rings: The Return of the King,2003.0,"Adventure,Drama,Fantasy",9.0,2157436.0,8.992312
60726,Schindler's List,1993.0,"Biography,Drama,History",9.0,1580941.0,8.989518
39255,The Godfather Part II,1974.0,"Crime,Drama",9.0,1490669.0,8.988886
24184,12 Angry Men,1957.0,"Crime,Drama",9.0,980205.0,8.983133
67006,The Lord of the Rings: The Fellowship of the Ring,2001.0,"Adventure,Drama,Fantasy",8.9,2199062.0,8.892722
183087,Inception,2010.0,"Action,Adventure,Sci-Fi",8.8,2804960.0,8.794499
72349,Fight Club,1999.0,"Crime,Drama,Thriller",8.8,2593969.0,8.794052


# On va utiliser le MultiLabelBinarizer afin de pouvoir encoder chaque genre.
# Pour ça, on va d'abord splitter la colonne genres afin de différencier tous les genres présents.

In [9]:
df['cat'] = df['genres'].str.split(",")

In [11]:
df['cat'].head()

0                         [Romance]
1        [Documentary, News, Sport]
2                               NaN
3    [Action, Adventure, Biography]
4                           [Drama]
Name: cat, dtype: object